# Production RAG Deployment

## Understanding Production Architecture

This notebook provides a **visual understanding** of how RAG systems are deployed to production. Instead of running deployment code (which requires Docker, cloud infrastructure, etc.), we focus on **architecture diagrams** and **conceptual explanations**.

---

### What You'll Learn

- **Full Production Architecture** - Complete deployment stack visualization
- **Component Breakdown** - Understand each part of the system
- **Request Flow** - How a query travels through the system
- **Configuration** - Example configurations (copy-paste ready)
- **Deployment Options** - Compare Local, Cloud, and Kubernetes
- **Monitoring** - Observability architecture
- **Security** - Production security checklist

### Why Visualization?

Production deployment involves many interconnected components:
- **FastAPI** servers (multiple instances)
- **Vector databases** (Pinecone, Weaviate, etc.)
- **LLM providers** (Ollama, OpenAI)
- **Caching layers** (Redis)
- **Load balancers**
- **Monitoring** (Prometheus, Grafana)

Trying to run all of this in a notebook isn't practical. Instead, we'll **visualize** and **understand** the architecture so you can deploy it yourself.


## Production RAG Architecture

Here's the complete architecture of a production RAG system:

```
+----------------------------------------------------------------------------------------+
|                          PRODUCTION RAG DEPLOYMENT                                     |
+----------------------------------------------------------------------------------------+
|                                                                                        |
|    +---------------------+      +---------------------+      +---------------------+   |
|    |    Web Application  |      |   Mobile App        |      |   API Clients       |   |
|    |                     |      |                     |      |                     |   |
|    |  React/Next.js      |      |   iOS/Android       |      |   curl/REST         |   |
|    +---------+-----------+      +---------+-----------+      +---------+-----------+   |
|              |                            |                            |               |
|              +----------------------------+----------------------------+               |
|                                           |                                            |
|                                           v                                            |
|                              +------------------------+                                |
|                              |    Load Balancer       |                                |
|                              |  (Nginx / Cloud LB)    |                                |
|                              |                        |                                |
|                              |  - Health checks       |                                |
|                              |  - SSL termination     |                                |
|                              |  - Rate limiting       |                                |
|                              +-----------+------------+                                |
|                                          |                                             |
|              +---------------------------+---------------------------+                 |
|              |                           |                           |                 |
|              v                           v                           v                 |
|    +---------------------+    +---------------------+    +---------------------+       |
|    |    API Instance 1   |    |    API Instance 2   |    |    API Instance N   |       |
|    |  +---------------+  |    |  +---------------+  |    |  +---------------+  |       |
|    |  |   FastAPI     |  |    |  |   FastAPI     |  |    |  |   FastAPI     |  |       |
|    |  |               |  |    |  |               |  |    |  |               |  |       |
|    |  |  - /query     |  |    |  |  - /query     |  |    |  |  - /query     |  |       |
|    |  |  - /health    |  |    |  |  - /health    |  |    |  |  - /health    |  |       |
|    |  |  - /embed     |  |    |  |  - /embed     |  |    |  |  - /embed     |  |       |
|    |  +-------+-------+  |    |  +-------+-------+  |    |  +-------+-------+  |       |
|    |          |          |    |          |          |    |          |          |       |
|    |  +-------+-------+  |    |  +-------+-------+  |    |  +-------+-------+  |       |
|    |  |  RAG Provider |  |    |  |  RAG Provider |  |    |  |  RAG Provider |  |       |
|    |  |               |  |    |  |               |  |    |  |               |  |       |
|    |  | - Query       |  |    |  | - Query       |  |    |  | - Query       |  |       |
|    |  | - Embed       |  |    |  | - Embed       |  |    |  | - Embed       |  |       |
|    |  | - Add docs    |  |    |  | - Add docs    |  |    |  | - Add docs    |  |       |
|    |  +-------+-------+  |    |  +-------+-------+  |    |  +-------+-------+  |       |
|    +----------+----------+    +----------+----------+    +----------+----------+       |
|               |                          |                          |                  |
|               +--------------------------+--------------------------+                  |
|                                          |                                             |
|                                          v                                             |
|    +----------------------------------------------------------------------+            |
|    |                              DATA LAYER                              |            |
|    |                                                                      |            |
|    |  +-----------------+    +-----------------+    +-----------------+   |            |
|    |  |   Vector DB     |    |  Document Store |    |      Cache      |   |            |
|    |  |                 |    |                 |    |                 |   |            |
|    |  | - Pinecone      |    | - S3/MinIO      |    |  - Redis        |   |            |
|    |  | - Weaviate      |    | - PostgreSQL    |    |  - Memcached    |   |            |
|    |  | - Qdrant        |    | - Elasticsearch |    |                 |   |            |
|    |  | - Milvus        |    |                 |    |  TTL: 1hr-24hr  |   |            |
|    |  +--------+--------+    +--------+--------+    +-------+---------+   |            |
|    +-----------+----------------------+---------------------+-------------+            |
|                |                      |                     |                          |
|                +----------------------+---------------------+                          |
|                                       |                                                |
|                                       v                                                |
|    +---------------------------------------------------------------------+             |
|    |                                LLM LAYER                            |             |
|    |                                                                     |             |
|    |  +-------------------------+      +-------------------------------+ |             |
|    |  |      Ollama             |      |         OpenAI / Claude       | |             |
|    |  |                         |      |                               | |             |
|    |  |  - Local GPU            |      |  - GPT-4, Claude, etc.        | |             |
|    |  |  - llama3.2             |      |  - API-based                  | |             |
|    |  |  - nomic-embed-text     |      |  - Pay-per-token              | |             |
|    |  +-------------------------+      +-------------------------------+ |             |
|    |                                                                     |             |
|    +---------------------------------------------------------------------+             |
|                                                                                        |
+----------------------------------------------------------------------------------------+

+------------------------------------------------------------------------------+
|                          OBSERVABILITY LAYER                                 |
|                                                                              |
|    +-------------+    +-------------+    +-------------+    +-------------+  |
|    |  Prometheus |<---|   Grafana   |    |     ELK     |<---|   Tracing   |  |
|    |             |    |             |    |    Stack    |    |  (Jaeger)   |  |
|    |  Metrics    |    |  Dashboards |    |    Logs     |    |             |  |
|    +-------------+    +-------------+    +-------------+    +-------------+  |
|                                                                              |
+------------------------------------------------------------------------------+
```

### Key Takeaways

1. **Multiple API Instances** - Run behind load balancer for high availability
2. **Vector DB** - Stores embeddings for semantic search
3. **Cache Layer** - Redis dramatically improves response times
4. **LLM Layer** - Can be local (Ollama) or cloud (OpenAI)
5. **Observability** - Essential for production debugging


## Component Breakdown

### 1. Load Balancer

- Distributes traffic across API instances
- Health checks (removes sick nodes)
- SSL/TLS termination
- Rate limiting
- Examples: Nginx, AWS ALB, Cloudflare

### 2. API Servers (FastAPI)

**Endpoints:**
- POST /query - Query RAG
- POST /add - Add documents
- GET /health - Health check
- POST /embed - Get embeddings

**Features:**
- Request validation (Pydantic)
- Authentication (API keys, JWT)
- Rate limiting
- Structured logging

### 3. RAG Provider

**Responsibilities:**
1. Text embedding (query)
2. Vector similarity search
3. Context retrieval (top-k)
4. Prompt engineering
5. LLM call
6. Response parsing

**Configuration:**
- Chunk size / overlap
- Retrieval top-k
- LLM model selection
- Temperature, max tokens

### 4. Cache Layer (Redis)

**What gets cached:**
- Query → Answer pairs
- Embeddings
- Document metadata

**Cache Strategies:**
- TTL (time-to-live): 1hr - 24hr
- LRU (least recently used)
- Invalidation on document update

**Impact:** 10-100x faster for cached queries

### 5. Vector Database

**Primary Operations:**
- Add vectors + metadata
- Similarity search (ANN)
- Filter by metadata
- Delete / update

**Options:**
- Pinecone (Managed cloud)
- Weaviate (Open source)
- Qdrant (Open source)
- Milvus (Open source)
- Chroma (Local/HuggingFace)
- pgvector (PostgreSQL extension)

### 6. LLM Layer

**Option A: Local (Ollama)**
- No API costs
- Data stays local
- Requires GPU infrastructure
- Slower than cloud

**Option B: Cloud (OpenAI/Claude)**
- Fast, scalable
- Latest models
- API costs
- Data leaves infrastructure

**Hybrid:** Local embedding + Cloud LLM


## Request Flow: How a Query Travels

Here's the step-by-step journey of a user query:

```
1. USER SUBMITS QUERY
   "What are the benefits of RAG?"
              |
              v
2. LOAD BALANCER
   - Receives request
   - Checks health of API instances
   - Forwards to healthy instance
              |
              v
3. FASTAPI SERVER
   - Validates request (Pydantic)
   - Checks authentication
   - Rate limiting check
   - Logs request
              |
              v
4. RAG PROVIDER

   4a. EMBED QUERY
       Input: "What are the benefits of RAG?"
       Output: [0.12, -0.34, 0.78, ...] (768-dim vector)
              |
              v
   4b. CACHE CHECK (Redis)
       Key: hash(embed("What are the benefits of RAG?"))
       |
       ├── HIT --> Return cached response (skip to 6)
       └── MISS --> Continue to vector search
              |
              v (Cache Miss)
   4c. VECTOR SEARCH
       Query: [0.12, -0.34, 0.78, ...]
       Returns: [chunk1, chunk2, chunk3, chunk4]
              |
              v
   4d. BUILD PROMPT
       System: "You are a helpful assistant..."
       Context: [1] "RAG provides fresh knowledge..."
               [2] "It reduces hallucinations..."
       Question: "What are the benefits of RAG?"
              |
              v
   4e. LLM GENERATION
       Prompt + Context --> LLM --> Response
       "RAG offers several key benefits: 1) Fresh knowledge..."
              |
              v
5. CACHE STORE
   Store: {query_embedding} --> {answer, sources}
   TTL: Set based on configuration (e.g., 1 hour)
              |
              v
6. RETURN RESPONSE
   {
     "answer": "RAG offers several key benefits...",
     "sources": [chunk1, chunk2, chunk3, chunk4],
     "cached": false,
     "latency_ms": 245
   }
```

### Timing Breakdown (Typical)

| Step | Time | % of Total |
|------|------|------------|
| Load Balancer | 2-5ms | 2% |
| API Validation | 1-2ms | 1% |
| Embed Query | 20-50ms | 15% |
| Cache Check | 1-2ms | 1% |
| Vector Search | 10-30ms | 10% |
| Build Prompt | 2-5ms | 2% |
| LLM Generation | 100-300ms | 65% |
| Cache Store | 2-5ms | 2% |
| **Total** | **150-400ms** | 100% |

**Note:** With cache hit, total time drops to ~5-10ms!


## Configuration Examples

These are **example configurations** you can copy and modify.

### Environment Variables (.env)

```bash
# ============================================================================
# RAG Application Configuration
# ============================================================================

# Application
APP_NAME=RAG-API
APP_VERSION=1.0.0
LOG_LEVEL=INFO
DEBUG=false

# Server
HOST=0.0.0.0
PORT=8000
WORKERS=4

# ============================================================================
# LLM Provider Configuration
# ============================================================================
# Choose ONE provider (ollama OR openai)

# Option 1: Ollama (Local)
LLM_PROVIDER=ollama
OLLAMA_HOST=http://ollama:11434
OLLAMA_MODEL=llama3.2
OLLAMA_EMBEDDING_MODEL=nomic-embed-text

# Option 2: OpenAI (Cloud)
# LLM_PROVIDER=openai
# OPENAI_API_KEY=sk-your-key-here
# OPENAI_MODEL=gpt-4o
# OPENAI_EMBEDDING_MODEL=text-embedding-3-small

# ============================================================================
# Vector Database Configuration
# ============================================================================
# Choose ONE vector DB

# Option 1: Chroma (Local file-based)
VECTOR_DB=chroma
VECTOR_DB_PATH=/data/chroma
VECTOR_DIMENSION=768

# Option 2: Pinecone (Cloud)
# VECTOR_DB=pinecone
# PINECONE_API_KEY=your-key
# PINECONE_ENVIRONMENT=us-west1
# PINECONE_INDEX=rag-index

# ============================================================================
# Cache Configuration (Redis)
# ============================================================================
REDIS_HOST=redis
REDIS_PORT=6379
REDIS_DB=0
CACHE_TTL=3600           # Cache expiration in seconds (1 hour)
CACHE_ENABLED=true

# ============================================================================
# RAG Configuration
# ============================================================================
CHUNK_SIZE=500
CHUNK_OVERLAP=50
RETRIEVAL_TOP_K=4
MAX_CONTEXT_CHARS=4000

# ============================================================================
# Security
# ============================================================================
API_KEY_ENABLED=true
CORS_ORIGINS=http://localhost:3000,https://yourdomain.com

# ============================================================================
# Monitoring
# ============================================================================
PROMETHEUS_ENABLED=true
PROMETHEUS_PORT=9090
TRACING_ENABLED=true
```

### Docker Compose (docker-compose.yml)

```yaml
version: '3.8'

services:
  # RAG API Service
  rag-api:
    build: .
    container_name: rag-api
    ports:
      - "8000:8000"
    environment:
      - LLM_PROVIDER=${LLM_PROVIDER}
      - OLLAMA_HOST=${OLLAMA_HOST}
      - VECTOR_DB=${VECTOR_DB}
      - REDIS_HOST=redis
    depends_on:
      - redis
      - ollama
    volumes:
      - ./data:/data
    restart: unless-stopped
    deploy:
      replicas: 2
      resources:
        limits:
          memory: 2G

  # Ollama (Local LLM)
  ollama:
    image: ollama/ollama:latest
    container_name: ollama
    ports:
      - "11434:11434"
    volumes:
      - ollama-data:/root/.ollama
    restart: unless-stopped

  # Redis (Caching)
  redis:
    image: redis:7-alpine
    container_name: rag-redis
    ports:
      - "6379:6379"
    volumes:
      - redis-data:/data
    restart: unless-stopped

  # Prometheus (Metrics)
  prometheus:
    image: prom/prometheus:latest
    container_name: rag-prometheus
    ports:
      - "9090:9090"
    volumes:
      - ./prometheus.yml:/etc/prometheus/prometheus.yml
    restart: unless-stopped

  # Grafana (Dashboards)
  grafana:
    image: grafana/grafana:latest
    container_name: rag-grafana
    ports:
      - "3000:3000"
    restart: unless-stopped
    depends_on:
      - prometheus

volumes:
  ollama-data:
  redis-data:
```


## Deployment Options Comparison

Choosing the right deployment option depends on your scale, budget, and expertise.

### Option Comparison

| Feature | Local Docker | Cloud (PaaS) | Kubernetes |
|---------|--------------|--------------|------------|
| **Complexity** | Low | Medium | High |
| **Cost** | $ (electricity) | $$ ($50-500/mo) | $$$$ ($500+/mo) |
| **Setup Time** | 1 hour | 1 day | 1 week |
| **Scalability** | Manual | Auto | Auto |
| **Maintenance** | You | Provider | You |
| **Best For** | Dev/Test | Startups | Enterprise |

### 1. Local Docker (Development/Testing)

```
Your Laptop/Server
  +--------------------------------+
  |    Docker Compose              |
  |                                |
  |  +-------+  +-------+  +------+|
  |  |  API  |  | Redis |  |Ollama||
  |  | 8000  |  | 6379  |  |11434 ||
  |  +-------+  +-------+  +------+|
  +--------------------------------+
```

**Pros:** Free, full control, works offline, easy to debug
**Cons:** Not production-ready, limited scalability, no redundancy
**Cost:** $0-50/month

### 2. Cloud PaaS (Render/Railway/Fly.io)

```
Cloud Provider
  +--------------------------------+
  |     Your Cloud Service         |
  |                                |
  |  +-------+  +-------+  +------+|
  |  |  API  |  | Redis |  |Ollama||
  |  |  URL  |  |  URL  |  |  URL ||
  |  +-------+  +-------+  +------+|
  +--------------------------------+
  Managed: Database, SSL, Deployments
```

**Pros:** Auto-scaling, SSL handled, Git-based deploys
**Cons:** Costs increase with usage, limited config
**Cost:** $50-500/month

### 3. Kubernetes (Enterprise)

```
Kubernetes Cluster
  +-------------------------------------------+
  |            Load Balancer                  |
  +-----------------+-------------------------+
  |                 |                         |
  +-----+     +-----+     +-----+     +-------+
  | Pod |     | Pod |     | Pod |     | Pod   |
  | API |     | API |     | API |     | API   |
  +-----+     +-----+     +-----+     +-------+
        |           |           |
    +---+---+   +---+---+   +---+---+
    | Redis |   |Vector|   |  LLM  |
    |Cache  |   |  DB  |   | Pool  |
    +---+---+   +---+---+   +-------+
```

**Pros:** Infinite scalability, high availability, full control
**Cons:** Steep learning, complex, expensive
**Cost:** $500-5000+/month

### Decision Guide

```
START: How many users?
  |
  +-- < 1000/day --> Local Docker (dev)
  |                    Ready for prod? --> No --> Cloud PaaS
  |
  +-- 1000-100K/day --> Cloud PaaS
  |
  +-- > 100K/day --> Budget?
                      |
                      +-- < $1000 --> Cloud PaaS + Optimize
                      +-- > $1000 --> Kubernetes

Recommended Path:
1. Start: Local Docker
2. Validate: Cloud PaaS (< $100/month)
3. Scale: Kubernetes (when needed)


## Monitoring & Observability

Production systems require comprehensive monitoring.

### The Three Pillars

1. **Metrics** (Prometheus + Grafana)
   - Request rate, error rate, latency
   - Memory, CPU, GPU usage
   - Cache hit ratio

2. **Logs** (ELK/Loki)
   - Application logs
   - Access logs
   - Error logs

3. **Traces** (Jaeger/Zipkin)
   - Request flow through services
   - Latency per component
   - Debug slow requests

### RAG-Specific Metrics

**Retrieval Metrics:**
- retrieval_latency_ms - Vector search time
- retrieval_k_matches - Documents retrieved
- retrieval_similarity_avg - Avg similarity score

**Generation Metrics:**
- generation_latency_ms - LLM response time
- tokens_used_prompt - Prompt token count
- tokens_used_completion - Response token count
- generation_cost_usd - API cost per request

**Quality Metrics:**
- context_precision - Retrieved chunk quality
- answer_faithfulness - Response accuracy

**Cache Metrics:**
- cache_hit_rate - % cached responses
- cache_latency_ms - Cache lookup time
- cache_size_mb - Redis memory usage

### Alerting Rules

```yaml
# High error rate
- alert: HighErrorRate
  expr: rate(errors_total[5m]) > 0.01
  severity: critical
  
# High latency (p95 > 2s)
- alert: HighLatency
  expr: histogram_quantile(0.95, rate(latency_seconds_bucket[5m])) > 2
  severity: warning

# Low cache hit rate (< 50%)
- alert: LowCacheHitRate
  expr: cache_hit_ratio < 0.5
  severity: warning
```


## Security Checklist

Production RAG systems need comprehensive security.

### Security Layers

1. **Network Security**
   - TLS/SSL (HTTPS)
   - Firewall rules
   - Private subnets (VPC)

2. **Application Security**
   - API Key authentication
   - JWT tokens (OAuth2)
   - Rate limiting
   - Input validation (Pydantic)

3. **Data Security**
   - Encryption at rest
   - Encryption in transit
   - Sensitive data masking

4. **LLM Security**
   - Prompt injection prevention
   - Output filtering
   - Content moderation
   - Cost limits (max tokens)

### Authentication

**API Keys (Simple):**
```python
API_KEYS = {"key_abc123": {"user": "user1", "rate_limit": 100}}

# Check API key
api_key = request.headers.get("X-API-Key")
if api_key not in API_KEYS:
    raise HTTPException(status_code=401)
```

**JWT Tokens (Advanced):**
```python
import jwt

# Verify JWT
payload = jwt.decode(token, SECRET_KEY, algorithms=["HS256"])
return payload["sub"]
```

### Security Checklist

- All dependencies updated
- No secrets in code
- HTTPS enabled
- API keys rotated
- Rate limiting configured
- Input validation implemented
- Encryption at rest + transit
- RBAC implemented
- Audit logging
- Anomaly detection


## Working Example: RAGProvider

While the deployment architecture is complex, let's demonstrate the **RAGProvider** class that makes it easy to query your RAG system.


In [1]:
# Import RAG provider
import sys
import os
import importlib.util

# Get project path
notebook_dir = os.path.dirname(os.path.abspath("__FILE__"))
project_root = os.path.dirname(notebook_dir)
providers_path = os.path.join(project_root, "docs", "_technical", "providers.py")

# Load module
spec = importlib.util.spec_from_file_location("providers", providers_path)
providers = importlib.util.module_from_spec(spec)
spec.loader.exec_module(providers)
RAGProvider = providers.RAGProvider

# Sample documents
documents = [
    "RAG stands for Retrieval-Augmented Generation. It combines a retrieval system with a generative AI model.",
    "RAG helps reduce hallucinations by grounding responses in retrieved context.",
    "Benefits of RAG include: fresh knowledge, reduced hallucinations, traceability, and cost-effective updates.",
    "Vector search works by converting text into embeddings and finding similar vectors.",
    "Embeddings are numerical representations of text that capture semantic meaning."
]

# Create RAG provider (uses Ollama by default)
rag = RAGProvider(provider="ollama")
rag.add_documents(documents)

print("RAG Provider ready!")
print("Try: rag.query('What is RAG?')")

RAG Provider ready!
Try: rag.query('What is RAG?')


## Query the RAG System

In [2]:
# Query the RAG system
result = rag.query("What is RAG?")

print("Question: What is RAG?")
print(f"\nAnswer: {result['answer']}")
print(f"\nSources retrieved: {len(result['sources'])}")
print(f"Cached: {result['cached']}")

Question: What is RAG?

Answer: RAG stands for Retrieval-Augmented Generation, a system that combines a retrieval system with a generative AI model.

Sources retrieved: 4
Cached: False


## Caching Demo

In [3]:
# First query - not cached
result1 = rag.query("What are the benefits of RAG?")
print(f"First query - Cached: {result1['cached']}")

# Second query - should be cached
result2 = rag.query("What are the benefits of RAG?")
print(f"Second query - Cached: {result2['cached']}")

# Clear cache
rag.clear_cache()
print("\nCache cleared!")

# Query again
result3 = rag.query("What are the benefits of RAG?")
print(f"After clear - Cached: {result3['cached']}")

First query - Cached: False
Second query - Cached: True

Cache cleared!
After clear - Cached: False


## Next Steps

### To Deploy to Production:

1. **Create FastAPI app** - Use the configuration examples above
2. **Containerize** - Use the Docker Compose template
3. **Deploy locally** - Test with docker-compose
4. **Deploy to cloud** - Render, Railway, or Kubernetes
5. **Set up monitoring** - Prometheus + Grafana
6. **Configure security** - API keys, TLS, RBAC

### Further Reading:

- [Production Deployment Guide](../docs/4-best-prodractices/production-deployment.md)
- [Caching Strategies](../docs/4-best-practices/caching.md)
- [Observability](../docs/4-best-practices/observability.md)
- [Security Considerations](../docs/4-best-practices/security-considerations.md)
